# Day 35 — Practice Questions
### Yes or No — logistic regression

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

| Part | What it asks you to do | How many |
|---|---|---|
| **A** | Work it out — pen and paper | 8 |
| **B** | Find the mistake in a short program | 4 |
| **C** | Write the code yourself | 5 |
| **D** | Challenge | 2 |

### How to work

**Part A needs a calculator at most.** Work it out, then run the cell to check.

Three questions to keep asking:

- Is this coefficient a probability, or a **log-odds**? (Always the second.)
- What is the **baseline**? Accuracy means nothing without it.
- Who chose the **cut-off**, and what does each mistake cost?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss

import warnings
warnings.simplefilter("ignore")
plt.rcParams["figure.figsize"] = (8, 3.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print("ready")

ready


---
# Part A — Work it out

A bank fits this model to predict whether a loan will be **repaid**:

```
z  =  0.42 × income_lakhs  −  0.65 × existing_loans  +  0.018 × credit_score  −  12.0
p  =  1 / (1 + e⁻ᶻ)
```

### A1 · The sigmoid, by hand

Work out `sigmoid(z)` for z = −2, 0 and +2. Do z = 0 without a calculator.

In [2]:
for z in [-2, 0, 2]:
    print(f"   sigmoid({z:>2}) = {sigmoid(z):.4f}")

   sigmoid(-2) = 0.1192
   sigmoid( 0) = 0.5000
   sigmoid( 2) = 0.8808


### A2 · Score one applicant

Income **8 lakhs**, **1** existing loan, credit score **720**.

Work out z, then p. Would the bank approve at a 0.5 cut-off?

In [3]:
def z_of(income, loans, score):
    return 0.42*income - 0.65*loans + 0.018*score - 12.0

z = z_of(8, 1, 720)
print(f"z = {z:.4f}")
print(f"p = {sigmoid(z):.4f}")

z = 3.6700
p = 0.9752


### A3 · What does +0.42 mean?

A colleague says *"income adds 0.42 to the probability of repaying."*

Explain why that is wrong, and write the correct sentence.

In [4]:
print("one extra lakh of income multiplies the odds by:", round(np.exp(0.42), 4))
print()
for inc in [5, 6, 7, 8]:
    p = sigmoid(z_of(inc, 1, 720))
    print(f"   income {inc} lakhs: p = {p:.4f},  odds = {p/(1-p):.4f}")

one extra lakh of income multiplies the odds by: 1.522

   income 5 lakhs: p = 0.9176,  odds = 11.1340
   income 6 lakhs: p = 0.9443,  odds = 16.9455
   income 7 lakhs: p = 0.9627,  odds = 25.7903
   income 8 lakhs: p = 0.9752,  odds = 39.2519


### A4 · Which column matters most?

The three coefficients are **+0.42**, **−0.65** and **+0.018**.

A colleague concludes that `credit_score` barely matters because 0.018 is tiny. Is he
right?

### A5 · Odds and probability

Fill in the gaps.

| probability | odds |
|---|---|
| 0.5 | ? |
| 0.8 | ? |
| ? | 9 |
| 0.25 | ? |

In [5]:
for p in [0.5, 0.8, 0.25]:
    print(f"   p = {p}  ->  odds = {p/(1-p):.4f}")
print()
for o in [9]:
    print(f"   odds = {o}  ->  p = {o/(1+o):.4f}")

   p = 0.5  ->  odds = 1.0000
   p = 0.8  ->  odds = 4.0000
   p = 0.25  ->  odds = 0.3333

   odds = 9  ->  p = 0.9000


### A6 · Log loss by hand

A model gives these four probabilities. Work out the log loss for each row, then the
average.

| | true answer | p the model gave |
|---|---|---|
| 1 | yes | 0.9 |
| 2 | yes | 0.6 |
| 3 | no | 0.2 |
| 4 | no | 0.8 |

In [6]:
truth = np.array([1., 1, 0, 0])
p = np.array([0.9, 0.6, 0.2, 0.8])

cost = np.where(truth == 1, -np.log(p), -np.log(1 - p))
for t, pp, c in zip(truth, p, cost):
    print(f"   truth {int(t)},  p = {pp}  ->  cost {c:.4f}")
print(f"\n   average log loss: {cost.mean():.4f}")
print(f"   sklearn         : {log_loss(truth, p):.4f}")

   truth 1,  p = 0.9  ->  cost 0.1054
   truth 1,  p = 0.6  ->  cost 0.5108
   truth 0,  p = 0.2  ->  cost 0.2231
   truth 0,  p = 0.8  ->  cost 1.6094

   average log loss: 0.6122
   sklearn         : 0.6122


### A7 · Is 97% accuracy good?

A model predicts whether an email is phishing. **2 emails in 100 are phishing.**
The model is **97% accurate**.

What is the most likely explanation, and what should you ask for instead?

### A8 · Choosing a cut-off

Two models, both at a 0.5 cut-off:

| | what it does | a false YES costs | a false NO costs |
|---|---|---|---|
| **A** | flags a tumour on a scan for a specialist to review | one unnecessary review, ~₹2,000 | a missed cancer |
| **B** | auto-blocks a credit card transaction as fraud | a customer stranded abroad | the bank refunds the fraud |

For each, should you move the cut-off up or down from 0.5?

---
# Part B — Find the mistake

Four short programs. Each runs cleanly and reports something misleading.

In [7]:
# ---------------------------------------------------------------
# 900 patients. B1 to B4 all use `scans`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3540)
n = 900

age       = rng.normal(55, 14, n).clip(20, 90).round(0)
lump_mm   = rng.gamma(2.0, 3.2, n).clip(0, 40).round(1)
family    = rng.binomial(1, 0.22, n).astype(float)

z = -6.3 + 0.035*age + 0.27*lump_mm + 0.70*family + rng.normal(0, 0.5, n)
malignant = (rng.uniform(0, 1, n) < sigmoid(z)).astype(int)

scans = pd.DataFrame({"age": age, "lump_mm": lump_mm,
                      "family_history": family, "malignant": malignant})
print("malignant rate:", round(scans.malignant.mean(), 4))
scans.head()

malignant rate: 0.1378


,age,lump_mm,family_history,malignant
0,65.0,6.1,1.0,0
1,72.0,25.6,0.0,1
2,60.0,6.4,0.0,0
3,62.0,4.5,0.0,0
4,62.0,5.8,0.0,1


### B1 · "Our model is 90% accurate"

In [8]:
X = scans[["age", "lump_mm", "family_history"]]
y = scans["malignant"]

X_train, X_hidden, y_train, y_hidden = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y)

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)

print(f"accuracy: {model.score(X_hidden, y_hidden):.4f}")
print()
print("REPORT: 'The screening model is 90% accurate. Recommend deployment.'")

accuracy: 0.9037

REPORT: 'The screening model is 90% accurate. Recommend deployment.'


### B2 · The cut-off nobody chose

In [9]:
pred = model.predict(X_hidden)              # <-- look here
tn, fp, fn, tp = confusion_matrix(y_hidden, pred).ravel()

print(f"we flag {tp + fp} patients for a biopsy")
print(f"we catch {tp} of the {tp + fn} real cancers")

we flag 21 patients for a biopsy
we catch 16 of the 37 real cancers


### B3 · "Family history is the biggest risk factor"

In [10]:
for name, c in zip(X.columns, model.coef_[0]):
    print(f"{name:<18} {c:>+8.4f}")
print()
print("REPORT: 'Family history is by far the strongest predictor.'")

age                 +0.0468
lump_mm             +0.2933
family_history      +0.6096

REPORT: 'Family history is by far the strongest predictor.'


### B4 · The split that threw away the cases

In [11]:
big = scans[scans.lump_mm > 14]             # 52 patients with a large lump
print("rows:", len(big), "  malignant:", int(big.malignant.sum()))

a, b, c, d = train_test_split(big[["age", "lump_mm", "family_history"]],
                              big.malignant, test_size=0.3, random_state=7)

m = LogisticRegression(max_iter=2000).fit(a, c)
print("accuracy on the hidden rows:", round(m.score(b, d), 4))

rows: 52   malignant: 33
accuracy on the hidden rows: 0.625


---
# Part C — Write the code

Five tasks on one dataset.

In [12]:
# ---------------------------------------------------------------
# 1,000 telecom customers. C1 to C5 all use `telco`.
# ---------------------------------------------------------------
rng = np.random.default_rng(3520)
N = 1000

tenure_months = rng.uniform(1, 72, N).round(0)
monthly_bill  = rng.normal(68, 22, N).clip(18, 130).round(2)
support_calls = rng.poisson(1.1, N).clip(0, 8).astype(float)
contract      = rng.choice(["monthly", "one_year", "two_year"], N, p=[.55, .28, .17])

bump = np.where(contract == "monthly", 1.15, np.where(contract == "one_year", 0.0, -0.95))
z = (-1.25 - 0.040*tenure_months + 0.021*monthly_bill + 0.38*support_calls + bump
     + rng.normal(0, 0.4, N))
churned = (rng.uniform(0, 1, N) < sigmoid(z)).astype(int)

telco = pd.DataFrame({"tenure_months": tenure_months, "monthly_bill": monthly_bill,
                      "support_calls": support_calls, "contract": contract,
                      "churned": churned})
telco.head()

,tenure_months,monthly_bill,support_calls,contract,churned
0,24.0,50.32,2.0,one_year,0
1,27.0,68.05,0.0,two_year,0
2,50.0,53.90,3.0,one_year,0
3,43.0,76.91,2.0,monthly,0
4,72.0,78.37,0.0,one_year,0


### C1 · Count before you fit

Step 1 for a classification problem is not a scatter plot — it is a **count**.

How many churned and how many did not? What is the baseline you have to beat? And does
the churn rate differ by contract type?

In [13]:
# your code here

### C2 · Fit it and read it in odds

One-hot the contract column, split with `stratify`, fit, and print each coefficient
**together with what it does to the odds**. Report accuracy on both halves, and the log
loss on the hidden rows.

In [14]:
# your code here

### C3 · Which column matters most?

Rank the five columns properly. Show raw and scaled side by side, and say which column
the raw numbers would have misled you about.

In [15]:
# your code here

### C4 · Choose a cut-off, and defend it

The retention team can call **at most 120** of the 300 hidden customers, and an offer
costs ₹600. A customer who churns is worth ₹9,000 in lost lifetime value.

Build the threshold table, then pick a cut-off and justify it **with the numbers**.

In [16]:
# your code here

### C5 · Score three real customers

For each, give the probability and say what you would do.

| | tenure | bill | support calls | contract |
|---|---|---|---|---|
| **A** | 3 months | ₹95 | 4 | monthly |
| **B** | 60 months | ₹55 | 0 | two_year |
| **C** | 18 months | ₹78 | 2 | one_year |

In [17]:
# your code here

---
# Part D — Challenge

Two tasks. Both are marked.

### D1 · Build logistic regression from scratch

Write `fit_logistic(X, y, rate, steps)` using no scikit-learn. It should:

- work for **any number of columns**, not just one
- use the gradient descent loop from yesterday, with the sigmoid in it
- report its progress so a reader can watch the log loss fall
- handle the scale problem — `monthly_bill` is in the tens while `support_calls` is 0 to 8

Then prove it matches `LogisticRegression(C=1e9)` on the telecom data.

*Reminder of the loop:*
```
p     = sigmoid(X @ w + b)
error = p - y                      # same as always
w    -= rate × (1/n) × (X.T @ error)
b    -= rate × (1/n) × error.sum()
```

In [18]:
# the telecom data from Part C, one-hot encoded, as a plain numpy array
encoded = pd.get_dummies(
    telco[["tenure_months", "monthly_bill", "support_calls", "contract"]],
    columns=["contract"], drop_first=True).astype(float)

demo_X = encoded.values
demo_y = telco.churned.values
print(demo_X.shape, demo_y.shape, list(encoded.columns))

(1000, 5) (1000,) ['tenure_months', 'monthly_bill', 'support_calls', 'contract_one_year', 'contract_two_year']


In [19]:
# your function here

### D2 · A screener that auto-rejects job applicants

A company has deployed the model below to filter applications. It reports:

> *"The screening model is **86% accurate** on applications it has never seen. Candidates
> the model scores below 0.5 are auto-rejected without human review. This saves the
> recruiting team roughly 90% of their reading time."*

The model is correctly fitted and correctly split. The accuracy figure is true.

**Your tasks:**

1. Run it and confirm the 86%.
2. Find **three separate problems** with deploying it as described.
3. Prove each one with numbers.
4. Write a short recommendation — not "don't use it", but what you would change.

In [20]:
rng = np.random.default_rng(3530)
A_ = 1200

years_experience   = rng.uniform(0, 12, A_).round(1)
skill_test         = rng.normal(58, 16, A_).clip(10, 100).round(0)
referral           = rng.binomial(1, 0.18, A_).astype(float)
portfolio_projects = rng.poisson(1.6, A_).clip(0, 8).astype(float)

z = (-6.5 + 0.16*years_experience + 0.052*skill_test + 1.35*referral
     + 0.22*portfolio_projects + rng.normal(0, 0.55, A_))
hired = (rng.uniform(0, 1, A_) < sigmoid(z)).astype(int)

apps = pd.DataFrame({"years_experience": years_experience, "skill_test": skill_test,
                     "referral": referral, "portfolio_projects": portfolio_projects,
                     "hired": hired})
apps.head()

,years_experience,skill_test,referral,portfolio_projects,hired
0,8.1,94.0,0.0,0.0,1
1,0.5,51.0,0.0,4.0,0
2,6.6,76.0,1.0,1.0,0
3,0.7,56.0,1.0,2.0,0
4,5.7,52.0,0.0,5.0,1


In [21]:
feat = ["years_experience", "skill_test", "referral", "portfolio_projects"]

a, b, c, d = train_test_split(apps[feat], apps.hired, test_size=0.3,
                              random_state=0, stratify=apps.hired)
screener = LogisticRegression(max_iter=2000).fit(a, c)

print(f"accuracy on applications it has never seen: {screener.score(b, d):.4f}")

accuracy on applications it has never seen: 0.8611


In [22]:
# your investigation here

---
## Before you hand it in

- Did you run every coefficient through `exp()` before describing it?
- Did you report accuracy **next to a baseline**?
- Did you use `.predict_proba()` rather than `.predict()`?
- Did you say **who chose the cut-off**, and what each mistake costs?
- Did you add `stratify=y` to the split?
- For every claim: does it carry "holding the other columns constant"?

Four numbers worth carrying away:

- A cancer screen at **90% accuracy** caught **16 of 37** tumours. At a cut-off of 0.05 it
  caught 36 — and its accuracy fell to 56%.
- A hiring screener at **86% accuracy** beat "reject everyone" by **4.4 points**, and
  auto-rejected **44 of 66** people who would have been hired.
- `tenure_months` had the **smallest** raw coefficient in the churn model (−0.040) and the
  **largest** influence once scaled.
- Our own gradient descent loop and `LogisticRegression(C=1e9)` agreed to **four decimal
  places** — the loop from yesterday, with one sigmoid added.

Answers will be shared after class.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*